# Model selection and class imbalance

Four things are measured here, each as a paired comparison over the same 25 folds: a reference
point for boosting on the final feature set, class weights, randomised hyperparameter search,
and the number of trees. Missingness indicators are re-checked at the end.

Sign convention: the variant under test goes first, so a **positive difference means the tested
variant is better**. In paired comparisons `±` is twice the standard error of the mean — the
threshold above which an effect is read as real. In stand-alone results it is the standard
deviation across folds.

In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from scipy.stats import loguniform, randint, uniform
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline

from src.config import DATA_5YEAR_PATH, EXCLUDED_FEATURES, RANDOM_STATE
from src.data import split_holdout
from src.pipelines import make_boosting_pipeline
from src.train import compare_pipelines, cross_validate_model

X_train, X_holdout, y_train, y_holdout = split_holdout(DATA_5YEAR_PATH)
feature_cols = [c for c in X_train.columns if c not in EXCLUDED_FEATURES]

In [2]:
def report(diff: np.ndarray) -> str:
    """Mean difference with twice its standard error."""
    se = diff.std(ddof=1) / np.sqrt(len(diff))
    return f"{diff.mean():+.4f} ± {2 * se:.4f}"


def show(diff: dict[str, np.ndarray]) -> None:
    """Print one line per metric: paired difference and how often it improved."""
    for name, d in diff.items():
        print(f"{name:15s} {report(d)}   folds improved: {(d > 0).sum()} / {len(d)}")


def summarize(res: dict[str, np.ndarray]) -> str:
    """Mean and standard deviation across folds, for stand-alone results."""
    return "   ".join(
        f"{name} {v.mean():.3f} ± {v.std(ddof=1):.3f}" for name, v in res.items()
    )

## Reference point

Default LightGBM on 63 features. The earlier figure of 0.817 was measured before `Attr21` was
excluded and does not apply to the current feature set.

In [3]:
base_model = LGBMClassifier(verbose=-1, random_state=RANDOM_STATE)
base_pipe = make_boosting_pipeline(base_model, feature_cols)

print(summarize(cross_validate_model(base_pipe, X_train, y_train)))

pr_auc 0.734 ± 0.038   precision_at_k 0.895 ± 0.062


**PR-AUC 0.734 ± 0.038, precision@top-3% 0.895 ± 0.062.** That is 10.5× the random-ranking
baseline of 0.070, and the gap to Altman (0.279) and logistic regression (0.348) is well above
the sum of two standard deviations.

At k = 26, P@3% of 0.895 means 23 bankrupts in the queue and 3 misses. One company is worth
1/26 = 0.038, so the metric cannot resolve anything smaller than that. Decisions below are
therefore made on PR-AUC; P@3% is reported as the business-facing translation.

## Class weights

`class_weight="balanced"` recomputes the weight on every `fit` from the training part of the
fold. A hard-coded `scale_pos_weight` would be a statistic of the full train set, part of which
sits in the validation fold.

Expectation stated before the run: no movement in P@3%, differences mostly exactly zero.

In [4]:
balanced_model = LGBMClassifier(
    verbose=-1, random_state=RANDOM_STATE, class_weight="balanced"
)
balanced_pipe = make_boosting_pipeline(balanced_model, feature_cols)

show(compare_pipelines(balanced_pipe, base_pipe, X_train, y_train))

pr_auc          +0.0012 ± 0.0078   folds improved: 12 / 25
precision_at_k  +0.0015 ± 0.0229   folds improved: 9 / 25


**PR-AUC +0.0012 ± 0.0078, 12 folds out of 25.** No effect.

Both metrics depend on ranking only. The main action of class weights — shifting predictions
towards the rare class — is monotone and leaves the ordering untouched; there is no threshold in
this task for it to act on. What remains is the secondary effect on split selection through the
loss, and with PR-AUC at 0.734 against a 0.070 baseline the signal is strong enough for splits
to be found without reweighting. Gradient boosting also reweights observations on its own:
well-predicted rows carry near-zero gradients and stop influencing later splits.

Weights add no information — 306 positives stay 306 positives. Of two configurations of equal
quality the simpler one is kept, so the final model carries no class weights.

`is_unbalance=True` applies the same ratio inside LightGBM instead of the sklearn wrapper. Same
mechanism, not measured separately.

## Resampling

SMOTE was not run. It interpolates synthetic positives between real ones, which is incompatible
with the native NaN handling in the boosting branch: imputation would have to be added, and the
comparison would no longer differ by a single factor. Interpolation across 63 dimensions with
tails up to 694 × q99 also produces financially impossible ratio combinations. Expected result:
no effect, for the same reason boosting already reweights its own observations.

The leakage mechanism is worth stating, since it is not the usual "resampling before the split".
A resampler changes which rows exist, and `sklearn.pipeline.Pipeline` treats it like any other
transformer, calling `transform` on the validation fold as well. Synthetic validation positives
are then interpolations between rows the model was trained on, and the positive rate in the fold
rises to 50%, so PR-AUC is computed on a different problem altogether.
`imblearn.pipeline.Pipeline` calls `fit_resample` during `fit` only, leaving validation intact.

## Hyperparameter search

`RandomizedSearchCV`, 40 iterations, 5 folds inside the search. Continuous distributions rather
than value lists: on a fixed budget a four-value list spends ten runs per value, a distribution
covers forty distinct points.

40 iterations rather than 500 because the winner is the maximum of noisy estimates, and the
larger the candidate pool the more of that maximum is luck with folds. `n_estimators` is fixed
at 400 rather than searched — it trades off against `learning_rate`, so searching both would
enumerate equivalent models. 400 gives the low end of the `learning_rate` range room to converge.

In [5]:
search_model = LGBMClassifier(n_estimators=400, verbose=-1, random_state=RANDOM_STATE)
search_pipe = make_boosting_pipeline(search_model, feature_cols)

param_distributions = {
    "model__num_leaves": randint(8, 32),
    "model__min_child_samples": randint(50, 200),
    "model__colsample_bytree": uniform(0.5, 0.5),
    "model__reg_lambda": loguniform(1e-2, 1e2),
    "model__learning_rate": loguniform(1e-2, 2e-1),
}

search = RandomizedSearchCV(
    estimator=search_pipe,
    param_distributions=param_distributions,
    n_iter=40,
    scoring="average_precision",
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    random_state=RANDOM_STATE,
    n_jobs=-1,
    refit=False,
)
search.fit(X_train, y_train);

In [6]:
cv_results = pd.DataFrame(search.cv_results_)
param_cols = [c for c in cv_results.columns if c.startswith("param_")]
cv_results[param_cols + ["mean_test_score", "std_test_score"]].sort_values(
    "mean_test_score", ascending=False
).head(10)

,param_model__colsample_bytree,param_model__learning_rate,param_model__min_child_samples,param_model__num_leaves,param_model__reg_lambda,mean_test_score,std_test_score
1,0.811782,0.031629,137,14,0.016860,0.723757,0.043097
16,0.836830,0.183878,181,10,1.092691,0.723483,0.035362
20,0.958361,0.157926,86,18,0.129085,0.722471,0.035751
9,0.693244,0.149385,92,9,6.173405,0.720810,0.036024
2,0.636328,0.041827,89,31,0.831589,0.720699,0.040073
25,0.819461,0.144524,111,22,0.626312,0.716072,0.038870
28,0.862627,0.044899,133,9,0.379732,0.715038,0.038534
3,0.696392,0.122395,122,17,3.914601,0.714122,0.030752
6,0.860316,0.057178,177,8,1.222907,0.713399,0.042244
13,0.826554,0.021357,193,28,0.094995,0.712687,0.039082


The top ten candidates span 0.713–0.724 with `std_test_score` between 0.031 and 0.043, so they
are indistinguishable from one another. No axis collapses: `num_leaves` in the top ten ranges
from 8 to 31, `learning_rate` from 0.021 to 0.184, `reg_lambda` from 0.017 to 6.17. Within these
ranges the problem is insensitive to hyperparameters, and the winning row reflects fold noise
rather than quality. No optimum sits at a range boundary, so the space needs no widening.

`best_score_` is not reported: it is the maximum of forty noisy estimates and optimistically
biased by construction. The selected configuration is re-measured below on the standard 5×5
scheme.

In [7]:
tuned_model = LGBMClassifier(n_estimators=400, verbose=-1, random_state=RANDOM_STATE)
tuned_pipe = make_boosting_pipeline(tuned_model, feature_cols)
tuned_pipe.set_params(**search.best_params_)

show(compare_pipelines(tuned_pipe, base_pipe, X_train, y_train))

pr_auc          -0.0131 ± 0.0081   folds improved: 6 / 25
precision_at_k  -0.0046 ± 0.0223   folds improved: 8 / 25


**PR-AUC −0.0131 ± 0.0081, 6 folds out of 25.** Above the threshold and consistent in direction:
the tuned configuration loses to the default in 19 folds out of 25.

Two things changed at once — the tuned parameters and four times as many trees. The tree count
is isolated next.

In [8]:
n400_model = LGBMClassifier(n_estimators=400, verbose=-1, random_state=RANDOM_STATE)
n400_pipe = make_boosting_pipeline(n400_model, feature_cols)

show(compare_pipelines(n400_pipe, base_pipe, X_train, y_train))

pr_auc          +0.0221 ± 0.0054   folds improved: 24 / 25
precision_at_k  +0.0262 ± 0.0170   folds improved: 12 / 25


**PR-AUC +0.0221 ± 0.0054, 24 folds out of 25.** Going from the default 100 trees to 400, with
everything else at defaults, gains four times the threshold.

Combining the two comparisons: trees contribute +0.022, the tuned model ends at −0.013, so the
tuned parameters cost roughly −0.035. That figure comes from subtracting two paired comparisons
rather than from a direct measurement, but both terms clear the threshold and point in opposite
directions, so the sign is safe.

The cause is visible in the search space. The default sits at or outside every boundary:
`min_child_samples` 20 against 50–200, `colsample_bytree` 1.0 against 0.5–1.0, `reg_lambda` 0
against 0.01–100, `num_leaves` 31 at the top edge. Every one of the forty candidates was more
constrained than the default. The ranges were chosen on the reasoning that 306 positives make
overfitting likely, but the data does not support it: at this signal strength the insurance
costs quality and does not pay for itself.

That leaves the question of whether 400 itself is justified — it was picked as headroom for the
search, not from the data.

In [9]:
n800_model = LGBMClassifier(n_estimators=800, verbose=-1, random_state=RANDOM_STATE)
n800_pipe = make_boosting_pipeline(n800_model, feature_cols)

show(compare_pipelines(n800_pipe, n400_pipe, X_train, y_train))

pr_auc          +0.0004 ± 0.0014   folds improved: 13 / 25
precision_at_k  +0.0015 ± 0.0070   folds improved: 3 / 25


**PR-AUC +0.0004 ± 0.0014, 13 folds out of 25.** Doubling the trees changes nothing.

The curve is now bounded on both sides: 100 is too few (+0.0221 moving to 400) and 800 is no
better than 400. Intermediate values were not tested — the 400–800 stretch is flat, and picking
a point inside a plateau on the third decimal would repeat the mistake already visible in the
search results. This is a one-parameter decision made by paired comparison under the reporting
rule, not a maximum over forty combinations, so winner's-curse bias is negligible here.

**Final configuration: default LightGBM with `n_estimators=400`.**

## Missingness indicators, re-check

The six indicators added nothing to boosting at default hyperparameters (−0.0004 ± 0.0006), but
that was measured at 100 trees. The configuration has changed, so the comparison is repeated.
The right-hand branch is the same 63 features without the indicator block.

In [10]:
no_flags_pipe = Pipeline(
    [("model", LGBMClassifier(n_estimators=400, verbose=-1, random_state=RANDOM_STATE))]
)

show(compare_pipelines(n400_pipe, no_flags_pipe, X_train[feature_cols], y_train))

pr_auc          -0.0000 ± 0.0008   folds improved: 6 / 25
precision_at_k  -0.0015 ± 0.0054   folds improved: 1 / 25


**PR-AUC −0.0000 ± 0.0008, 6 folds out of 25.** The conclusion holds.

This is the tightest threshold in the project: the branches differ by six columns and almost all
shared noise cancels, so the claim is stronger than "small effect" — there is no effect down to
thousandths. LightGBM routes NaN at every split towards whichever side lowers the loss and
recovers missingness on its own; an explicit flag adds nothing.

The block stays in the factory. The cost on the boosting side is measured and equal to zero, and the two factories stay symmetric in structure. Whether the indicators earn their place in the linear branch has not been measured: median imputation replaces NaN with a value the model cannot tell apart from a real one, so missingness is expected to be recoverable only from an explicit flag there — but that is an argument, not a result.

## Summary

In [11]:
final_pipe = n400_pipe
print(summarize(cross_validate_model(final_pipe, X_train, y_train)))

pr_auc 0.756 ± 0.039   precision_at_k 0.922 ± 0.042


| Model | PR-AUC | precision@top-3% |
|---|---|---|
| Random ranking | 0.070 | 0.070 |
| Altman Z' (1983) | 0.279 ± 0.049 | 0.442 ± 0.099 |
| Logistic regression | 0.348 ± 0.039 | 0.492 ± 0.081 |
| LightGBM, defaults | 0.734 ± 0.038 | 0.895 ± 0.062 |
| LightGBM + `class_weight="balanced"` | +0.0012 ± 0.0078 vs defaults | +0.0015 ± 0.0229 |
| LightGBM after randomised search | −0.0131 ± 0.0081 vs defaults | −0.0046 ± 0.0223 |
| **LightGBM, `n_estimators=400`** | 0.756 ± 0.039 | 0.922 ± 0.042 |

The two middle rows are given as paired differences against the default model rather than as
stand-alone means: differences of that size fall below the sum of two standard deviations and
cannot be read from independently reported figures.

The final model gains roughly 0.48 PR-AUC over Altman against a combined spread of about 0.09,
and roughly 0.41 over logistic regression. Both clear the threshold several times over.

All figures come from cross-validation on train. The holdout has not been touched.